# Deze notebook legt stapsgewijs uit hoe je AI pose detectie kan gebruiken om salamanders te detecteren en te croppen.

Installeer eerst de nodige packages in je virtual pip of conda environment.

PS C:\Users\sebastiaan_verbessel\Salamander_detectie> Set-ExecutionPolicy -Scope Process -ExecutionPolicy Bypass
PS C:\Users\sebastiaan_verbessel\Salamander_detectie> & C:/Users/sebastiaan_verbessel/Salamander_detectie/.venv/Scripts/Activate.ps1

```powershell
.venv/Scripts/activate.bat
```

```powershell
pip install -r requirements.txt
```

```powershell
labelme --labels labels.txt
```

Print your lables for a certain json file:

In [16]:
import json

# Load the JSON (replace with your file path)
with open("C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen/01478742.json", "r") as f:
    data = json.load(f)

# Extract all points
points_data = []
for shape in data["shapes"]:
    label = shape["label"]
    shape_type = shape["shape_type"]
    coords = shape["points"]
    points_data.append({
        "label": label,
        "shape_type": shape_type,
        "points": coords
    })

for p in points_data:
    print(p)

{'label': 'headbasis', 'shape_type': 'point', 'points': [[492.258064516129, 343.8225806451613]]}
{'label': 'rightforelegbasis', 'shape_type': 'point', 'points': [[490.3225806451613, 301.241935483871]]}
{'label': 'righthindlegbasis', 'shape_type': 'point', 'points': [[297.4193548387097, 202.5322580645161]]}
{'label': 'tailbasis', 'shape_type': 'point', 'points': [[254.83870967741936, 219.30645161290323]]}
{'label': 'lefthindlegbasis', 'shape_type': 'point', 'points': [[282.5806451612903, 248.3387096774194]]}
{'label': 'leftforelegbasis', 'shape_type': 'point', 'points': [[441.93548387096774, 328.98387096774195]]}
{'label': 'salamander', 'shape_type': 'rectangle', 'points': [[52.25806451612903, 149.6290322580645], [561.9354838709677, 415.43548387096774]]}


## 1. Convert your json labels to YOLO text labels

In [17]:
import json
import os
from glob import glob

# Directory containing LabelMe JSON files
input_dir = "C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen"  # change to your folder path
#output_dir = "yolo_labels"         # where YOLO txt files will go
#os.makedirs(output_dir, exist_ok=True)

# Define keypoint order (must match your model training config)
keypoint_labels = [
    "headbasis",
    "rightforelegbasis",
    "righthindlegbasis",
    "tailbasis",
    "lefthindlegbasis",
    "leftforelegbasis"
]

for json_path in glob(os.path.join(input_dir, "*.json")):
    with open(json_path, "r") as f:
        data = json.load(f)

    img_w = data["imageWidth"]
    img_h = data["imageHeight"]

    # --- Extract bounding box for salamander ---
    bbox_shape = next(s for s in data["shapes"] if s["label"] == "salamander")
    (x1, y1), (x2, y2) = bbox_shape["points"]

    bbox_cx = ((x1 + x2) / 2) / img_w
    bbox_cy = ((y1 + y2) / 2) / img_h
    bbox_w = abs(x2 - x1) / img_w
    bbox_h = abs(y2 - y1) / img_h

    # --- Extract keypoints ---
    keypoints = []
    for kp_label in keypoint_labels:
        shape = next((s for s in data["shapes"] if s["label"] == kp_label), None)
        if shape:
            px, py = shape["points"][0]
            keypoints.extend([px / img_w, py / img_h, 2.0]) # 2.0 for visible
        else:
            # If missing, set to 0,0
            keypoints.extend([0.0, 0.0, 0.0])

    # --- Create YOLOv10 line ---
    class_index = 0  # salamander = class 0
    yolo_line = [class_index, bbox_cx, bbox_cy, bbox_w, bbox_h] + keypoints
    yolo_str = " ".join(f"{v:.6f}" for v in yolo_line)

    # --- Save txt file ---
    base_name = os.path.splitext(os.path.basename(data["imagePath"]))[0]
    txt_path = os.path.join(input_dir, base_name + ".txt")
    with open(txt_path, "w") as f:
        f.write(yolo_str + "\n")

    print(f"Saved: {txt_path}")

print("✅ All JSON files converted to YOLOv10 pose format.")

Saved: C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen\01478742.txt
Saved: C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen\download (1).txt
Saved: C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen\download.txt
Saved: C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen\images (1).txt
Saved: C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen\images (2).txt
Saved: C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen\images (3).txt
Saved: C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen\images.txt
Saved: C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen\istockphoto-505619753-612x612.txt
Saved: C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen\japanese-fire-bellied-newt4.txt
Saved: C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen\newt_image.txt
Saved: C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen\salamander_web.txt
✅ All JSON files converted to Y

## 2. Create a YOLO dataset (80% for training, 20% for validation)

In [29]:
import os
import random
import shutil
from glob import glob
import yaml

# ==== CONFIG ====
images_dir = "C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen"  # your source images directory
labels_dir = images_dir  # your YOLO pose txt labels directory
dataset_root = "salamander-pose"  # output dataset root folder
train_ratio = 0.8
num_keypoints = 6   # update if different
dims = 3            # 2 for x,y or 3 for x,y,visible

# ==== CREATE DIRS ====
train_img_dir = os.path.join(dataset_root, "images", "train")
val_img_dir = os.path.join(dataset_root, "images", "val")
train_lbl_dir = os.path.join(dataset_root, "labels", "train")
val_lbl_dir = os.path.join(dataset_root, "labels", "val")

for d in [train_img_dir, val_img_dir, train_lbl_dir, val_lbl_dir]:
    os.makedirs(d, exist_ok=True)

# ==== SPLIT FILES ====
all_images = sorted(glob(os.path.join(images_dir, "*.*")))
random.shuffle(all_images)
split_index = int(len(all_images) * train_ratio)
train_images = all_images[:split_index]
val_images = all_images[split_index:]

def copy_files(img_list, img_dst, lbl_dst):
    for img_path in img_list:
        fname = os.path.basename(img_path)
        name_no_ext = os.path.splitext(fname)[0]
        lbl_path = os.path.join(labels_dir, name_no_ext + ".txt")
        if not os.path.exists(lbl_path):
            print(f"⚠ Warning: No label for {fname}, skipping.")
            continue
        shutil.copy(img_path, os.path.join(img_dst, fname))
        shutil.copy(lbl_path, os.path.join(lbl_dst, name_no_ext + ".txt"))

copy_files(train_images, train_img_dir, train_lbl_dir)
copy_files(val_images, val_img_dir, val_lbl_dir)

# ==== CREATE YAML FILE ====
yaml_dict = {
    "path": dataset_root,
    "train": "images/train",
    "val": "images/val",
    "test": "",
    "kpt_shape": [num_keypoints, dims],
    "flip_idx": list(range(num_keypoints)),  # update if you have symmetric keypoints
    "nc": 1,  # number of classes
    "names": ["salamander"] # class names
}

yaml_path = os.path.join(dataset_root, "dataset.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(yaml_dict, f, sort_keys=False)

print(f"✅ Dataset created at '{dataset_root}'")
print(f"YAML config saved to: {yaml_path}")


✅ Dataset created at 'salamander-pose'
YAML config saved to: salamander-pose\dataset.yaml


## 3. Train a YOLO-pose model

In [27]:
#%pip install ultralytics

In [30]:
import ultralytics
from ultralytics import YOLO

# Load the YOLOv10 pose model
model = YOLO("yolov8n-pose.pt")  

# Train the model on the salamander dataset
model.train(
    data="salamander-pose/dataset.yaml",  # path to your dataset YAML file
    epochs=10,  # adjust as needed
    imgsz=640,   # input image size
    batch=8,    # batch size
    # device="0"   # use GPU 0, change if needed
    project="results/salamander_runs",  # custom folder
    name="exp1_pose"                    # subfolder name
)

Ultralytics 8.3.179  Python-3.12.9 torch-2.8.0+cpu CPU (13th Gen Intel Core(TM) i5-1335U)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=salamander-pose/dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp1_pose, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12

train: Scanning C:\Users\sebastiaan_verbessel\Salamander_detectie\salamander-pose\labels\train... 10 images, 0 backgrounds, 0 corrupt: 100%|██████████| 10/10 [00:00<?, ?it/s]

train: New cache created: C:\Users\sebastiaan_verbessel\Salamander_detectie\salamander-pose\labels\train.cache
val: Fast image access  (ping: 0.10.0 ms, read: 52.30.0 MB/s, size: 8.5 KB)



c:\Users\sebastiaan_verbessel\Salamander_detectie\.venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
val: Scanning C:\Users\sebastiaan_verbessel\Salamander_detectie\salamander-pose\labels\val... 1 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1/1 [00:00<?, ?it/s]

val: New cache created: C:\Users\sebastiaan_verbessel\Salamander_detectie\salamander-pose\labels\val.cache
Plotting labels to results\salamander_runs\exp1_pose\labels.jpg... 


...

                   all          1          1       0.81          1      0.995      0.204      0.446          1      0.497     0.0995



10 epochs completed in 0.009 hours.
Optimizer stripped from results\salamander_runs\exp1_pose\weights\last.pt, 6.4MB
Optimizer stripped from results\salamander_runs\exp1_pose\weights\best.pt, 6.4MB

Validating results\salamander_runs\exp1_pose\weights\best.pt...
Ultralytics 8.3.179  Python-3.12.9 torch-2.8.0+cpu CPU (13th Gen Intel Core(TM) i5-1335U)
YOLOv8n-pose summary (fused): 81 layers, 3,088,301 parameters, 0 gradients, 8.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00, 11.84it/s]


                   all          1          1       0.73          1      0.995      0.416       0.73          1      0.995      0.199
Speed: 1.1ms preprocess, 73.0ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to results\salamander_runs\exp1_pose


ultralytics.utils.metrics.PoseMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000024D96EDDDF0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(P)', 'F1-Confidence(P)', 'Precision-Confidence(P)', 'Recall-Confidence(P)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.0340

In [31]:
# Train the model on the salamander dataset
model.train(
    data="salamander-pose/dataset.yaml",  # path to your dataset YAML file
    epochs=100,  # adjust as needed
    imgsz=640,   # input image size
    batch=8,    # batch size
    # device="0"   # use GPU 0, change if needed
    project="results/salamander_runs",  # custom folder
    name="exp2_pose"                    # subfolder name
)

Ultralytics 8.3.179  Python-3.12.9 torch-2.8.0+cpu CPU (13th Gen Intel Core(TM) i5-1335U)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=salamander-pose/dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp2_pose, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=1

train: Scanning C:\Users\sebastiaan_verbessel\Salamander_detectie\salamander-pose\labels\train.cache... 10 images, 0 backgrounds, 0 corrupt: 100%|██████████| 10/10 [00:00<?, ?it/s]

val: Fast image access  (ping: 0.10.0 ms, read: 65.40.0 MB/s, size: 8.5 KB)



c:\Users\sebastiaan_verbessel\Salamander_detectie\.venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
val: Scanning C:\Users\sebastiaan_verbessel\Salamander_detectie\salamander-pose\labels\val.cache... 1 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1/1 [00:00<?, ?it/s]

Plotting labels to results\salamander_runs\exp2_pose\labels.jpg... 


...

                   all          1          1      0.969          1      0.995     0.0995      0.969          1      0.995      0.398



100 epochs completed in 0.097 hours.
Optimizer stripped from results\salamander_runs\exp2_pose\weights\last.pt, 6.4MB
Optimizer stripped from results\salamander_runs\exp2_pose\weights\best.pt, 6.4MB

Validating results\salamander_runs\exp2_pose\weights\best.pt...
Ultralytics 8.3.179  Python-3.12.9 torch-2.8.0+cpu CPU (13th Gen Intel Core(TM) i5-1335U)
YOLOv8n-pose summary (fused): 81 layers, 3,088,301 parameters, 0 gradients, 8.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00, 12.29it/s]


                   all          1          1      0.955          1      0.995      0.597      0.955          1      0.995      0.398
Speed: 0.6ms preprocess, 75.8ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to results\salamander_runs\exp2_pose


ultralytics.utils.metrics.PoseMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000024D94693A40>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(P)', 'F1-Confidence(P)', 'Precision-Confidence(P)', 'Recall-Confidence(P)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.0340

In [19]:
model_path = "./results/salamander_runs/exp2_pose/weights/best.pt"


image_path = "C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen/01478742.jpeg"
results = model.predict(source=image_path, save=True, save_txt=True)

NameError: name 'model' is not defined

In [42]:
import cv2
model_path = "./results/salamander_runs/exp2_pose/weights/best.pt"
image_path = "C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen/01478742.jpeg"

model = YOLO(model_path)

results = model(image_path)[0]

# Iterate through keypoints and print them
for result in results:
    # Access the keypoint data directly from the tensor and convert it to a list
    if result.keypoints is not None:
        keypoints_list = result.keypoints.data.tolist()
        
        for keypoint in keypoints_list:
            # Each keypoint in keypoints_list is likely a list itself, e.g., [x, y, confidence]
            print(f"Keypoint: {keypoint}")


image 1/1 C:\Users\sebastiaan_verbessel\Salamander_detectie\afbeeldingen\01478742.jpeg: 448x640 3 salamanders, 73.0ms
Speed: 2.1ms preprocess, 73.0ms inference, 1.2ms postprocess per image at shape (1, 3, 448, 640)
Keypoint: [[387.3702392578125, 249.43092346191406, 0.43530017137527466], [383.0011291503906, 237.34950256347656, 0.4912409782409668], [384.2545166015625, 277.68597412109375, 0.5429341793060303], [392.0443115234375, 241.24215698242188, 0.6463170647621155], [365.00299072265625, 261.2163391113281, 0.500176727771759], [372.4327087402344, 248.09617614746094, 0.5201058387756348]]
Keypoint: [[370.3097839355469, 288.06378173828125, 0.45209234952926636], [332.40020751953125, 258.033935546875, 0.522905707359314], [353.835205078125, 291.1862487792969, 0.5687482953071594], [345.4305419921875, 264.56402587890625, 0.6875392198562622], [326.02618408203125, 282.2149658203125, 0.5479859113693237], [339.07037353515625, 273.22271728515625, 0.6197198033332825]]
Keypoint: [[662.2769165039062, 2

In [1]:
import cv2
from ultralytics import YOLO  # assuming Ultralytics YOLO
import numpy as np

model_path = "./results/salamander_runs/exp2_pose/weights/best.pt"
image_path = "C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen/01478742.jpeg"

img = cv2.imread(image_path)
model = YOLO(model_path)

results = model(image_path)[0]  # first result

if results.keypoints is not None:
    # Convert tensor to numpy for easier indexing
    keypoints_array = results.keypoints.data.cpu().numpy()  # shape: (num_instances, num_keypoints, 3)
    
    for instance_kpts in keypoints_array:  # loop over each detected object
        for idx, (x, y, conf) in enumerate(instance_kpts):
            print(f"Keypoint {idx}: ({x}, {y}), conf={conf}")
            if conf > 0.5:  # optional confidence filter
                cv2.putText(img, str(idx), (int(x), int(y)), cv2.FONT_HERSHEY_SIMPLEX, 
                            0.5, (0, 255, 0), 2)

cv2.imshow("img", img)
cv2.waitKey(0)
#cv2.destroyAllWindows()



image 1/1 C:\Users\sebastiaan_verbessel\Salamander_detectie\afbeeldingen\01478742.jpeg: 448x640 2 salamanders, 109.3ms
Speed: 6.0ms preprocess, 109.3ms inference, 6.9ms postprocess per image at shape (1, 3, 448, 640)
Keypoint 0: (367.3825378417969, 185.07862854003906), conf=0.43622201681137085
Keypoint 1: (378.88720703125, 166.4191436767578), conf=0.5053519606590271
Keypoint 2: (400.8189697265625, 201.51023864746094), conf=0.4963076114654541
Keypoint 3: (419.3103332519531, 201.5138702392578), conf=0.6859739422798157
Keypoint 4: (392.60491943359375, 208.792724609375), conf=0.516789972782135
Keypoint 5: (390.2745361328125, 210.31317138671875), conf=0.503899097442627
Keypoint 0: (590.3272094726562, 181.89071655273438), conf=0.45746684074401855
Keypoint 1: (547.2376098632812, 148.51768493652344), conf=0.4994722306728363
Keypoint 2: (571.8568115234375, 191.791748046875), conf=0.5573173761367798
Keypoint 3: (558.6854248046875, 174.3267822265625), conf=0.6745214462280273
Keypoint 4: (536.755

-1

In [ ]:
import cv2
from PIL import Image, ImageOps
import cv2
import numpy as np


def visualize_annotations(image_path, annotation_path, output_img_path=None):
    """Visualize markup from YOLOv8-pose format with image."""

    # Load the image
    image = cv2.imread(image_path)

    #Rotate the image 180° over its caenter
    #image = cv2.rotate(image, cv2.ROTATE_180)

    # Get the dimensions of the rotated image
    img_height, img_width, _ = image.shape

    # Read annotations
    with open(annotation_path, "r") as f:
        annotations = f.readlines()

    for ann in annotations:
        ann_values = list(map(float, ann.split()))

        # Get bbox coordinates
        bbox_coords = get_bbox_coco_coords(img_height, img_width, ann_values)

        # Get keypoint coordinates
        keypoints_coords = get_keypoints(ann_values, img_width, img_height)

        # Draw bounding box
        x, y, w, h = bbox_coords
        cv2.rectangle(image, (x, y), (x + w, y + h), (255, 0, 0), 2)

        # Draw key points
        image = keypoints_viz(image, keypoints_coords)

    # Save or display
    if output_img_path is None:
        output_img_path = "viz3.jpg"

    cv2.imwrite(output_img_path, image)
def keypoints_viz(image, keypoints_coords, vis_color=None):
    """A function to visualize keypoints in the image.

    Parameters
    ----------
    image : numpy.array
        2d numpy array represent OpenCV image
    keypoints_coords: list
        list of keypoints coords, each element of this list is
        tuple in form of (x, y, v), where
        x - coordinate of keypoint in pixels of image
        y - coordinate of keypoint in pixels of image
        v - status of visibility from MS COCO dataset
        v=0: not labeled (in which case x=y=0),
        v=1: labeled but not visible,
        and v=2: labeled and visible.
    vis_color : dict, optional
        dictionary that matches the color of a keypoint
        when it is rendered, depending on its visibility status,
        by default None

    Returns
    -------
    _type_
        _description_
    """
    if vis_color is None:
        vis_color = {0: (255, 0, 0), 1: (0, 0, 255), 2: (0, 255, 0)}

    for point_coords in keypoints_coords:
        x, y, v = point_coords
        color = vis_color[v]
        image = cv2.circle(image, (x, y), radius=5, color=color, thickness=-1)
    return image


def get_keypoints(ann_values, img_width, img_height):
    keypoints_flat = ann_values[5:]
    keypoints_coords = []
    for i in range(0, len(keypoints_flat), 3):
        x = keypoints_flat[i]
        y = keypoints_flat[i + 1]
        v = int(keypoints_flat[i + 2])
        x, y = unnormalize_coords(x, y, img_width, img_height)
        keypoints_coords.append((x, y, v))
    return keypoints_coords


def get_bbox_coco_coords(img_height, img_width, ann_values):
    x_center, y_center, w_norm, h_norm = ann_values[1:5]
    w = int(w_norm * img_width)
    h = int(h_norm * img_height)
    x_min = int((x_center - w_norm / 2) * img_width)
    y_min = int((y_center - h_norm / 2) * img_height)
    return (x_min, y_min, w, h)


def unnormalize_coords(x, y, width, height):
    """Get the original image coordinates from the normalized
    coordinates.

    Parameters
    ----------
    x : float
        normalized coordinate on the X axis
    y : float
       normalized coordinate on the Y axis
    width : int
        image width in pixels
    height : int
        image height in pixels

    Returns
    -------
    x, y
       original coordinates x, y in pixels of the image.
    """
    x = int(x * width)
    y = int(y * height)
    return x, y


if __name__ == "__main__":
    image_path = 'C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen/01478742.jpeg'
    annotation_path = 'C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen/01478742.txt'

    # Visualize the keypoints labeling
    visualize_annotations(image_path, annotation_path)


In [ ]:
import cv2


def visualize_annotations(image_path, annotation_path, output_img_path=None):
    """Visualize markup from YOLOv8-pose format with image."""

    # Load the image
    image = cv2.imread(image_path)

    # Rotate the image 180° over its center
    #image = cv2.rotate(image, cv2.ROTATE_180)

    # Get the dimensions of the rotated image
    img_height, img_width, _ = image.shape

    # Read annotations
    with open(annotation_path, "r") as f:
        annotations = f.readlines()

    for ann in annotations:
        ann_values = list(map(float, ann.split()))

        # Get bbox coordinates
        bbox_coords = get_bbox_coco_coords(img_height, img_width, ann_values)

        # Get keypoint coordinates
        keypoints_coords = get_keypoints(ann_values, img_width, img_height)

        # Draw bounding box
        x, y, w, h = bbox_coords
        cv2.rectangle(image, (x, y), (x + w, y + h), (255, 0, 0), 2)

        # Draw key points
        image = keypoints_viz(image, keypoints_coords)

    # Save or display
    if output_img_path is None:
        output_img_path = "viz4.jpg"

    cv2.imwrite(output_img_path, image)
def keypoints_viz(image, keypoints_coords, vis_color=None):
    """Visualize keypoints and draw main axis lines."""
    if vis_color is None:
        vis_color = {0: (255, 0, 0), 1: (0, 0, 255), 2: (0, 255, 0)}

    # Draw keypoints
    for x, y, v in keypoints_coords:
        color = vis_color[v]
        image = cv2.circle(image, (x, y), radius=5, color=color, thickness=-1)

    # Draw main axis lines
    main_axis_pairs = [(0, 3), (1, 5), (2, 4)]
    axis_color = (0, 255, 255)  # yellow lines
    axis_thickness = 2

    for p1, p2 in main_axis_pairs:
        x1, y1, v1 = keypoints_coords[p1]
        x2, y2, v2 = keypoints_coords[p2]
        # Draw line only if both keypoints are labeled
        if v1 != 0 and v2 != 0:
            image = cv2.line(image, (x1, y1), (x2, y2), axis_color, axis_thickness)

    return image


def get_keypoints(ann_values, img_width, img_height):
    """Get the bounding box coordinates from the YOLOv8 pose
    format for one object (from a single row of the annotation file).

    Parameters
    ----------
    ann_values : list
        list with values from the YOLOv8 pose format for one object.
        (see docs).
    img_height : int
        image height in pixels
    img_width : int
        image width in pixels

    Returns
    -------
    keypoints_coords: list
        list of keypoints coords, each element of this list is
        tuple in form of (x, y, v), where
        x - coordinate of keypoint in pixels of image
        y - coordinate of keypoint in pixels of image
        v - status of visibility from MS COCO dataset
        v=0: not labeled (in which case x=y=0),
        v=1: labeled but not visible,
        and v=2: labeled and visible.
    """
    keypoints_x = ann_values[5::3]
    keypoints_y = ann_values[6::3]
    keypoints_v = ann_values[7::3]

    keypoints_coords = []
    for x, y, v in zip(keypoints_x, keypoints_y, keypoints_v):
        x, y = unnormalize_coords(x, y, img_width, img_height)
        v = int(v)
        keypoints_coords.append((x, y, v))

    return keypoints_coords


def get_bbox_coco_coords(img_height, img_width, ann_values):
    """Get the bounding box coordinates from the YOLOv8 pose
    format for one object (from a single row of the annotation file).

    Parameters
    ----------
    img_height : int
        image height in pixels
    img_width : int
        image width in pixels
    ann_values : list
        list with values from the YOLOv8 pose format for one object.
        (see docs).

    Returns
    -------
    list
        list with bbox coordinates in MS COCO format
    """
    # YOLO bbox format normalized [x_center, y_center, width, height]
    x_center, y_center, width, height = ann_values[1:5]
    x_min = x_center - width / 2
    y_min = y_center - height / 2
    x_min, y_min = unnormalize_coords(x_min, y_min, img_width, img_height)
    width, height = unnormalize_coords(width, height, img_width, img_height)

    # MS COCO bbox format [x_min, y_min, width, height]
    bbox_coords = (x_min, y_min, width, height)

    return bbox_coords


def unnormalize_coords(x, y, width, height):
    """Get the original image coordinates from the normalized
    coordinates.

    Parameters
    ----------
    x : float
        normalized coordinate on the X axis
    y : float
       normalized coordinate on the Y axis
    width : int
        image width in pixels
    height : int
        image height in pixels

    Returns
    -------
    x, y
       original coordinates x, y in pixels of the image.
    """
    x = int(x * width)
    y = int(y * height)
    return x, y


if __name__ == "__main__":
    image_path = 'C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen/01478742.jpeg'
    annotation_path = 'C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen/01478742.txt'

    # Visualize the keypoints labeling
    visualize_annotations(image_path, annotation_path)


Now for all images

In [17]:
import cv2
import os
import glob

def visualize_annotations(image_path, annotation_path, output_img_path=None):
    """Visualize markup from YOLOv8-pose format with image rotated 180° over its center --> adaptation: we do not do the 180° flip."""
    image = cv2.imread(image_path)
    #image = cv2.rotate(image, cv2.ROTATE_180)

    img_height, img_width, _ = image.shape

    with open(annotation_path, "r") as f:
        annotations = f.readlines()

    for ann in annotations:
        ann_values = list(map(float, ann.split()))
        bbox_coords = get_bbox_coco_coords(img_height, img_width, ann_values)
        keypoints_coords = get_keypoints(ann_values, img_width, img_height)

        x, y, w, h = bbox_coords
        cv2.rectangle(image, (x, y), (x + w, y + h), (255, 0, 0), 2)
        image = keypoints_viz(image, keypoints_coords)

    if output_img_path is None:
        output_img_path = os.path.splitext(image_path)[0] + "_viz.jpg"
    cv2.imwrite(output_img_path, image)

def keypoints_viz(image, keypoints_coords, vis_color=None):
    if vis_color is None:
        vis_color = {0: (255, 0, 0), 1: (0, 0, 255), 2: (0, 255, 0)}

    for x, y, v in keypoints_coords:
        color = vis_color[v]
        image = cv2.circle(image, (x, y), radius=5, color=color, thickness=-1)

    main_axis_pairs = [(0, 3), (1, 5), (2, 4)]
    axis_color = (0, 255, 255)
    axis_thickness = 2
    for p1, p2 in main_axis_pairs:
        x1, y1, v1 = keypoints_coords[p1]
        x2, y2, v2 = keypoints_coords[p2]
        if v1 != 0 and v2 != 0:
            image = cv2.line(image, (x1, y1), (x2, y2), axis_color, axis_thickness)
    return image

def get_keypoints(ann_values, img_width, img_height):
    keypoints_x = ann_values[5::3]
    keypoints_y = ann_values[6::3]
    keypoints_v = ann_values[7::3]
    keypoints_coords = []
    for x, y, v in zip(keypoints_x, keypoints_y, keypoints_v):
        x, y = unnormalize_coords(x, y, img_width, img_height)
        v = int(v)
        keypoints_coords.append((x, y, v))
    return keypoints_coords

def get_bbox_coco_coords(img_height, img_width, ann_values):
    x_center, y_center, width, height = ann_values[1:5]
    x_min = x_center - width / 2
    y_min = y_center - height / 2
    x_min, y_min = unnormalize_coords(x_min, y_min, img_width, img_height)
    width, height = unnormalize_coords(width, height, img_width, img_height)
    return (x_min, y_min, width, height)

def unnormalize_coords(x, y, width, height):
    x = int(x * width)
    y = int(y * height)
    return x, y

if __name__ == "__main__":
    input_dir = r"C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen"
    output_dir = os.path.join(input_dir, "outputs")
    os.makedirs(output_dir, exist_ok=True)

    for img_path in glob.glob(os.path.join(input_dir, "*.jpeg")):
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        ann_path = os.path.join(input_dir, base_name + ".txt")
        if os.path.exists(ann_path):
            out_path = os.path.join(output_dir, base_name + "_viz.jpg")
            visualize_annotations(img_path, ann_path, out_path)
            print(f"Processed {base_name}")
        else:
            print(f"Annotation not found for {base_name}")


Processed 01478742
Processed download (1)
Processed download
Processed images (1)
Processed images (2)
Processed images (3)
Processed images
Processed newt_image


In [18]:
'''
import cv2
import numpy as np

def visualize_annotations(image_path, annotation_path, output_img_path=None):
    """Visualize markup from YOLOv8-pose format, rotate 180°, 
       rectify image based on keypoints, and draw annotations."""

    # Load and rotate the image 180°
    image = cv2.imread(image_path)
    image = cv2.rotate(image, cv2.ROTATE_180)

    img_height, img_width, _ = image.shape

    # Read YOLO annotations
    with open(annotation_path, "r") as f:
        annotations = f.readlines()

    for ann in annotations:
        ann_values = list(map(float, ann.split()))

        # Get keypoint coordinates
        keypoints_coords = get_keypoints(ann_values, img_width, img_height)

        # === RECTIFY BASED ON KEYPOINTS ===
        # KP indices from your list (0-based indexing):
        # 1 → index 1, 2 → index 2, 4 → index 4, 5 → index 5
        bl = keypoints_coords[1][:2]  # bottom-left
        br = keypoints_coords[2][:2]  # bottom-right
        tr = keypoints_coords[4][:2]  # top-right
        tl = keypoints_coords[5][:2]  # top-left

        src_pts = np.array([bl, br, tr, tl], dtype=np.float32)

        # Compute rectangle size
        width_bottom = np.linalg.norm(np.array(br) - np.array(bl))
        width_top = np.linalg.norm(np.array(tr) - np.array(tl))
        rect_width = int(max(width_bottom, width_top))

        height_left = np.linalg.norm(np.array(tl) - np.array(bl))
        height_right = np.linalg.norm(np.array(tr) - np.array(br))
        rect_height = int(max(height_left, height_right))

        dst_pts = np.array([
            [0, rect_height - 1],     # bottom-left
            [rect_width - 1, rect_height - 1],  # bottom-right
            [rect_width - 1, 0],      # top-right
            [0, 0]                    # top-left
        ], dtype=np.float32)

        # Perspective transform
        M = cv2.getPerspectiveTransform(src_pts, dst_pts)
        rectified_img = cv2.warpPerspective(image, M, (rect_width, rect_height))

        # Replace image with rectified for annotation drawing
        image = rectified_img
        img_height, img_width, _ = image.shape

        # Draw bounding box
        bbox_coords = get_bbox_coco_coords(img_height, img_width, ann_values)
        x, y, w, h = bbox_coords
        # cv2.rectangle(image, (x, y), (x + w, y + h), (255, 0, 0), 2)

        # Draw keypoints
        image = keypoints_viz(image, keypoints_coords)

    # Save
    if output_img_path is None:
        output_img_path = "rectified_viz.jpg"
    cv2.imwrite(output_img_path, image)

def get_keypoints(ann_values, img_width, img_height):
    """Get the bounding box coordinates from the YOLOv8 pose
    format for one object (from a single row of the annotation file).

    Parameters
    ----------
    ann_values : list
        list with values from the YOLOv8 pose format for one object.
        (see docs).
    img_height : int
        image height in pixels
    img_width : int
        image width in pixels

    Returns
    -------
    keypoints_coords: list
        list of keypoints coords, each element of this list is
        tuple in form of (x, y, v), where
        x - coordinate of keypoint in pixels of image
        y - coordinate of keypoint in pixels of image
        v - status of visibility from MS COCO dataset
        v=0: not labeled (in which case x=y=0),
        v=1: labeled but not visible,
        and v=2: labeled and visible.
    """
    keypoints_x = ann_values[5::3]
    keypoints_y = ann_values[6::3]
    keypoints_v = ann_values[7::3]

    keypoints_coords = []
    for x, y, v in zip(keypoints_x, keypoints_y, keypoints_v):
        x, y = unnormalize_coords(x, y, img_width, img_height)
        v = int(v)
        keypoints_coords.append((x, y, v))

    return keypoints_coords


def get_bbox_coco_coords(img_height, img_width, ann_values):
    """Get the bounding box coordinates from the YOLOv8 pose
    format for one object (from a single row of the annotation file).

    Parameters
    ----------
    img_height : int
        image height in pixels
    img_width : int
        image width in pixels
    ann_values : list
        list with values from the YOLOv8 pose format for one object.
        (see docs).

    Returns
    -------
    list
        list with bbox coordinates in MS COCO format
    """
    # YOLO bbox format normalized [x_center, y_center, width, height]
    x_center, y_center, width, height = ann_values[1:5]
    x_min = x_center - width / 2
    y_min = y_center - height / 2
    x_min, y_min = unnormalize_coords(x_min, y_min, img_width, img_height)
    width, height = unnormalize_coords(width, height, img_width, img_height)

    # MS COCO bbox format [x_min, y_min, width, height]
    bbox_coords = (x_min, y_min, width, height)

    return bbox_coords


def unnormalize_coords(x, y, width, height):
    """Get the original image coordinates from the normalized
    coordinates.

    Parameters
    ----------
    x : float
        normalized coordinate on the X axis
    y : float
       normalized coordinate on the Y axis
    width : int
        image width in pixels
    height : int
        image height in pixels

    Returns
    -------
    x, y
       original coordinates x, y in pixels of the image.
    """
    x = int(x * width)
    y = int(y * height)
    return x, y


if __name__ == "__main__":
    image_path = 'C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen/01478742.jpeg'
    annotation_path = 'C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen/01478742.txt'

    # Visualize the keypoints labeling
    visualize_annotations(image_path, annotation_path)

'''    

'\nimport cv2\nimport numpy as np\n\ndef visualize_annotations(image_path, annotation_path, output_img_path=None):\n    """Visualize markup from YOLOv8-pose format, rotate 180°, \n       rectify image based on keypoints, and draw annotations."""\n\n    # Load and rotate the image 180°\n    image = cv2.imread(image_path)\n    image = cv2.rotate(image, cv2.ROTATE_180)\n\n    img_height, img_width, _ = image.shape\n\n    # Read YOLO annotations\n    with open(annotation_path, "r") as f:\n        annotations = f.readlines()\n\n    for ann in annotations:\n        ann_values = list(map(float, ann.split()))\n\n        # Get keypoint coordinates\n        keypoints_coords = get_keypoints(ann_values, img_width, img_height)\n\n        # === RECTIFY BASED ON KEYPOINTS ===\n        # KP indices from your list (0-based indexing):\n        # 1 → index 1, 2 → index 2, 4 → index 4, 5 → index 5\n        bl = keypoints_coords[1][:2]  # bottom-left\n        br = keypoints_coords[2][:2]  # bottom-right\n 

In [19]:
import cv2
import numpy as np
import os
import glob

def visualize_annotations(image_path, annotation_path, output_img_path=None):
    """Visualize markup from YOLOv8-pose format, rotate  (adpatation: we do not do the 180° flip, only for first image 01478742.jpeg the 180° flip was required), 
       rectify image based on keypoints, and draw annotations."""

    # Load and rotate the image 180°
    image = cv2.imread(image_path)
    #image = cv2.rotate(image, cv2.ROTATE_180)

    img_height, img_width, _ = image.shape

    # Read YOLO annotations
    with open(annotation_path, "r") as f:
        annotations = f.readlines()

    for ann in annotations:
        ann_values = list(map(float, ann.split()))

        # Get keypoint coordinates
        keypoints_coords = get_keypoints(ann_values, img_width, img_height)

        # === RECTIFY BASED ON KEYPOINTS ===
        bl = keypoints_coords[1][:2]  # bottom-left
        br = keypoints_coords[2][:2]  # bottom-right
        tr = keypoints_coords[4][:2]  # top-right
        tl = keypoints_coords[5][:2]  # top-left

        src_pts = np.array([bl, br, tr, tl], dtype=np.float32)

        # --- Add 5px buffer --- (new addition)
        # Compute center of rectangle
        center = np.mean(src_pts, axis=0)

        # Expand each corner outward by 5 pixels
        buffer = 20
        expanded_pts = []
        for pt in src_pts:
            # Vector from center to corner
            v = pt - center
            # Normalize vector
            norm = np.linalg.norm(v)
            if norm == 0:
                expanded_pts.append(pt)  # fallback if degenerate
            else:
                expanded_pts.append(pt + (v / norm) * buffer)

        expanded_pts = np.array(expanded_pts, dtype=np.float32)

        # expanded_pts order matches [bl, br, tr, tl]
        bl_e, br_e, tr_e, tl_e = expanded_pts

        # --- Compute new rectangle size using expanded points ---
        width_bottom = np.linalg.norm(br_e - bl_e)
        width_top = np.linalg.norm(tr_e - tl_e)
        rect_width = int(max(width_bottom, width_top))

        height_left = np.linalg.norm(tl_e - bl_e)
        height_right = np.linalg.norm(tr_e - br_e)
        rect_height = int(max(height_left, height_right))

        dst_pts = np.array([
            [0, rect_height - 1],     # bottom-left
            [rect_width - 1, rect_height - 1],  # bottom-right
            [rect_width - 1, 0],      # top-right
            [0, 0]                    # top-left
        ], dtype=np.float32)

        # --- (end new addition)
        
        # # Compute rectangle size
        # width_bottom = np.linalg.norm(np.array(br) - np.array(bl))
        # width_top = np.linalg.norm(np.array(tr) - np.array(tl))
        # rect_width = int(max(width_bottom, width_top))

        # height_left = np.linalg.norm(np.array(tl) - np.array(bl))
        # height_right = np.linalg.norm(np.array(tr) - np.array(br))
        # rect_height = int(max(height_left, height_right))

        # dst_pts = np.array([
        #     [0, rect_height - 1],     # bottom-left
        #     [rect_width - 1, rect_height - 1],  # bottom-right
        #     [rect_width - 1, 0],      # top-right
        #     [0, 0]                    # top-left
        # ], dtype=np.float32)

        # # Perspective transform
        # M = cv2.getPerspectiveTransform(src_pts, dst_pts)

        

        # Perspective transform
        M = cv2.getPerspectiveTransform(expanded_pts, dst_pts) # new addition
        rectified_img = cv2.warpPerspective(image, M, (rect_width, rect_height))

        # Replace image with rectified for annotation drawing
        image = rectified_img
        img_height, img_width, _ = image.shape

        # Draw keypoints
        image = keypoints_viz(image, keypoints_coords)

    # Save
    if output_img_path is None:
        output_img_path = os.path.splitext(image_path)[0] + "_rectified.jpg"
    cv2.imwrite(output_img_path, image)


def get_keypoints(ann_values, img_width, img_height):
    keypoints_x = ann_values[5::3]
    keypoints_y = ann_values[6::3]
    keypoints_v = ann_values[7::3]

    keypoints_coords = []
    for x, y, v in zip(keypoints_x, keypoints_y, keypoints_v):
        x, y = unnormalize_coords(x, y, img_width, img_height)
        v = int(v)
        keypoints_coords.append((x, y, v))
    return keypoints_coords


def unnormalize_coords(x, y, width, height):
    x = int(x * width)
    y = int(y * height)
    return x, y


def keypoints_viz(image, keypoints_coords, vis_color=None):
    if vis_color is None:
        vis_color = {0: (255, 0, 0), 1: (0, 0, 255), 2: (0, 255, 0)}

    for x, y, v in keypoints_coords:
        color = vis_color.get(v, (255, 255, 255))
        image = cv2.circle(image, (x, y), radius=5, color=color, thickness=-1)

    main_axis_pairs = [(0, 3), (1, 5), (2, 4)]
    axis_color = (0, 255, 255)
    axis_thickness = 2
    for p1, p2 in main_axis_pairs:
        x1, y1, v1 = keypoints_coords[p1]
        x2, y2, v2 = keypoints_coords[p2]
        if v1 != 0 and v2 != 0:
            image = cv2.line(image, (x1, y1), (x2, y2), axis_color, axis_thickness)
    return image


if __name__ == "__main__":
    input_folder = r"C:/Users/sebastiaan_verbessel/Salamander_detectie/afbeeldingen"
    output_folder = os.path.join(input_folder, "rectified_outputs")
    os.makedirs(output_folder, exist_ok=True)

    jpeg_files = glob.glob(os.path.join(input_folder, "*.jpeg"))

    for img_file in jpeg_files:
        base_name = os.path.splitext(os.path.basename(img_file))[0]
        txt_file = os.path.join(input_folder, base_name + ".txt")
        if os.path.exists(txt_file):
            out_path = os.path.join(output_folder, base_name + "_rectified_buffer.jpg")
            visualize_annotations(img_file, txt_file, out_path)
            print(f"Processed {base_name}")
        else:
            print(f"No matching annotation for {img_file}")


Processed 01478742
Processed download (1)
Processed download
Processed images (1)
Processed images (2)
Processed images (3)
Processed images
Processed newt_image
